# NVIDIA Nemotron Model Reasoning Challenge - End-to-End Pipeline

This notebook contains the complete pipeline to reproduce the SFT fine-tuning and package the LoRA adapter for the competition. It automates:
1. Dependency installation.
2. Data generation and reasoning corpus building.
3. SFT training client initiation.
4. Local packaging and tensor conversion (QR/SVD for Mamba, expert unfusing) into the required `submission.zip` format.

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q tinker tinker-cookbook safetensors torch transformers jinja2 pydantic huggingface_hub

## 2. Credentials Configuration

For security, retrieve your credentials securely from Kaggle User Secrets. Ensure you add `HF_TOKEN`, `TINKER_API_KEY`, and `KAGGLE_API_TOKEN` to your Kaggle Secrets.

In [ ]:
import os
import json
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    TINKER_API_KEY = user_secrets.get_secret("TINKER_API_KEY")
    KAGGLE_API_TOKEN = user_secrets.get_secret("KAGGLE_API_TOKEN")
    print("Successfully loaded credentials from Kaggle User Secrets.")
except Exception as e:
    print("Kaggle User Secrets not found. Falling back to manual entry (please fill below if needed).")
    HF_TOKEN = "YOUR_HF_TOKEN"
    TINKER_API_KEY = "YOUR_TINKER_API_KEY"
    KAGGLE_API_TOKEN = "YOUR_KAGGLE_API_TOKEN"

# Create env.json
env_data = {
    "HF_TOKEN": HF_TOKEN,
    "TINKER_API_KEY": TINKER_API_KEY,
    "KAGGLE_API_TOKEN": KAGGLE_API_TOKEN
}
with open("env.json", "w") as f:
    json.dump(env_data, f, indent=4)

# Log in to Hugging Face (needed for gated Nemotron model weights access)
from huggingface_hub import login
login(token=HF_TOKEN)

# Set environment variable for Tinker SDK
os.environ["TINKER_API_KEY"] = TINKER_API_KEY

## 3. Data Processing & Augmentation

In [ ]:
# Generate reasoning files, synthetic augmentations, and build corpus
!python3 reasoning.py
!python3 augmentation.py
!python3 corpus.py

## 4. Run SFT Training

In [ ]:
# Execute the SFT training script
!python3 train_sft.py

## 5. Convert & Package Adapter

Run the local converter to SVD-merge and package the adapter into `submission.zip`.

In [ ]:
# Run the packaging script
!python3 convert_and_package.py

## 6. Verification

Check that `submission.zip` was successfully created and contains the required files in its root directory.

In [ ]:
import os
import zipfile

zip_name = "submission.zip"
if os.path.exists(zip_name):
    print(f"Success! {zip_name} created. Size: {os.path.getsize(zip_name) / 1e6:.2f} MB")
    with zipfile.ZipFile(zip_name, 'r') as z:
        print("Contents:")
        for info in z.infolist():
            print(f"  {info.filename} ({info.file_size / 1e6:.2f} MB)")
else:
    print("Error: submission.zip not found!")